<a href="https://colab.research.google.com/github/Beefimaru/GB-Auto-Stocks/blob/main/Copy_of_KERETABEKAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install supabase


## Securely Store Supabase Credentials

To securely store your Supabase credentials, use Google Colab's **Secrets Manager**. Click the '🔑' icon in the left sidebar, then add two new secrets:

1.  **Name:** `SUPABASE_URL`
    **Value:** Your Supabase Project URL (e.g., `https://abcdefg.supabase.co`)
2.  **Name:** `SUPABASE_KEY`
    **Value:** Your Supabase `anon` public key (found in Project Settings > API)

Make sure to enable "Notebook access" for these secrets.

In [ ]:
from google.colab import userdata

url = None
key = None

try:
    # Retrieve the URL and key from the secrets manager
    url = userdata.get('SUPABASE_URL')
    key = userdata.get('SUPABASE_KEY')
    print("Credentials loaded from Colab Secrets.")
except userdata.SecretNotFoundError as e:
    print(f"❌ Error: {e}. Please ensure 'SUPABASE_URL' and 'SUPABASE_KEY' are set in Colab Secrets and have Notebook Access enabled.")
except Exception as e:
    print(f"❌ An unexpected error occurred while loading secrets: {e}")

Credentials loaded from Colab Secrets.


In [ ]:
# Verify the loaded URL and Key (key will be masked)
print(f"Loaded Supabase URL: {url}")
print(f"Loaded Supabase Key (masked): {key[:5]}...{key[-5:]}")

# Ensure both are not empty
if not url:
    print("Error: SUPABASE_URL is empty. Please check your Colab Secret.")
if not key:
    print("Error: SUPABASE_KEY is empty. Please check your Colab Secret.")

Loaded Supabase URL: https://cajakefmyyksuungnfcn.supabase.co
Loaded Supabase Key (masked): eyJhb...q8iBA


In [ ]:
import os
from supabase import create_client, Client

# 1. Connect to your Supabase Warehouse
# Using url and key retrieved from Colab's Secrets Manager

if url and key and url != 'your_url_here' and key != 'your_key_here':
    try:
        supabase: Client = create_client(url, key)
        print("✅ Successfully connected to Supabase!")

        # 2. Create a test car listing
        # We use a standard local market example to ensure data types match your SQL setup
        test_car = {
            "make": "Perodua",
            "model": "Myvi 1.5 H",
            "year": 2020,
            "mileage": 45000,
            "price_body": 42000,
            "price_otr": 44500,
            "location_name": "Petaling Jaya",
            "listing_url": "https://www.mudah.my/test-myvi-listing",
            "seller_phone": "60123456789",
            "platform_source": "test_script"
        }

        # 3. Push the data to the 'listings' table
        try:
            response = supabase.table("listings").insert(test_car).execute()
            print("✅ Test car successfully injected into database!")
            print(response.data)
        except Exception as e:
            print(f"❌ Failed to insert data: {e}")

    except Exception as e:
        print(f"❌ Connection failed: {e}")
else:
    print("⚠️ Supabase URL or Key is missing or still a placeholder. Please check your Colab Secrets and re-run this cell after setting them correctly.")

✅ Successfully connected to Supabase!
❌ Failed to insert data: {'message': 'duplicate key value violates unique constraint "listings_listing_url_key"', 'code': '23505', 'hint': None, 'details': 'Key (listing_url)=(https://www.mudah.my/test-myvi-listing) already exists.'}


### Resolving Supabase Row-Level Security (RLS) Error

The error message `new row violates row-level security policy for table "listings"` indicates that your Supabase database has Row-Level Security (RLS) enabled on the `listings` table, and the current `anon` user (associated with your `SUPABASE_KEY`) does not have permission to insert data.

To fix this, you need to configure an RLS policy in your Supabase dashboard:

1.  **Go to your Supabase Project Dashboard**: Navigate to your project on [app.supabase.com](https://app.supabase.com).
2.  **Navigate to the `Authentication` > `Policies` section**.
3.  **Select the `listings` table**.
4.  **Enable Row Level Security**: If not already enabled, click the button to "Enable Row Level Security".
5.  **Create a New Policy for `INSERT` operations**:
    *   Click `New Policy`.
    *   Choose `From Scratch`.
    *   **Name**: Give it a descriptive name, e.g., `Allow anon insert`.
    *   **Forced**: Keep this checked.
    *   **Target Roles**: Select `anon`.
    *   **Using expression**: Leave this blank or set it to `TRUE` (to allow all `anon` inserts).
    *   **With check expression**: Leave this blank.
    *   Click `Review` and then `Create policy`.

After setting up this policy, re-run the previous code cell, and the data insertion should succeed.